# Automation and practical scripts
* python is one of THE most popular languages for small, everyday automation - "write a quick script that does the boring thing for me"
* the reason is a combo of two things: python's "batteries included" standard library (remember lesson 05 - json, csv, pathlib, os, all built in) plus a huge ecosystem of great third-party tools on top
* this lesson is a grab-bag of genuinely useful building blocks: command-line tools, running shell commands from python, organizing files automatically, and scraping websites
* think of this as the "go automate something in your own life" lesson - almost everything here you can copy, tweak a little bit, and actually use for real

## building a real command-line tool with argparse
* `argparse` is standard library (no pip install needed) and is THE classic way to build a proper CLI tool in python
* it gives you named arguments/flags (`--name value`), required vs optional arguments, default values, automatic type conversion, and a free `--help` message
* normally you would call `parser.parse_args()` with no arguments and it automatically reads `sys.argv` (the real command line arguments the user typed)
* since we are inside a notebook and not a real terminal, we instead pass an explicit list of strings into `parse_args([...])` to simulate what a user would have typed - this is only a notebook trick, in a real `.py` script you almost never pass anything into `parse_args()`

In [1]:
import argparse  # standard library, no install needed - THE classic way to build CLI tools in python

# lets build a tiny tool that greets someone, a certain number of times
parser = argparse.ArgumentParser(description="a tiny greeting tool")  # description shows up in --help

# --name is a REQUIRED argument (no default= given, so argparse insists on it)
parser.add_argument("--name", required=True, help="who should we greet?")

# --times is OPTIONAL, has a default value, and type=int automatically converts the string from the command line into a real int for us
# (without type=int, "3" would stay the string "3" and "3" * "hello" would just crash)
parser.add_argument("--times", type=int, default=1, help="how many times to greet (default: 1)")

# in a REAL .py script you would just write:
#     args = parser.parse_args()
# and argparse reads sys.argv (the actual arguments the user typed after "python myscript.py") automatically
# here in the notebook we simulate a user typing "--name Alex --times 3" by passing an explicit list instead:
args = parser.parse_args(["--name", "Alex", "--times", "3"])

print("parsed args object:", args)
print("args.name:", args.name)
print("args.times:", args.times, type(args.times))  # really an int, not the string "3"

for _ in range(args.times):
    print(f"hello {args.name}!")

# lets also try it with ONLY the required argument, to see the default kick in
args_default = parser.parse_args(["--name", "Mia"])
print("\nwith default --times:", args_default.times)

parsed args object: Namespace(name='Alex', times=3)
args.name: Alex
args.times: 3 <class 'int'>
hello Alex!
hello Alex!
hello Alex!

with default --times: 1


In [2]:
# argparse also generates a --help message for us automatically, completely free, based on the add_argument() calls above
# normally typing "python myscript.py --help" in a real terminal prints this and exits
# in the notebook we can trigger the same text with format_help(), without it wanting to exit our whole notebook process
print(parser.format_help())

# what happens if a REQUIRED argument is missing? argparse normally prints an error and exits the whole program (sys.exit)
# that behavior is perfect for a real terminal tool, but would kill our notebook kernel, so we just describe it here instead of running it:
# parser.parse_args(["--times", "5"])   # --> would print "error: the following arguments are required: --name" and exit(2)

usage: ipykernel_launcher.py [-h] --name NAME [--times TIMES]

a tiny greeting tool

options:
  -h, --help     show this help message and exit
  --name NAME    who should we greet?
  --times TIMES  how many times to greet (default: 1)



## running external commands with subprocess
* sometimes the thing you want automated already exists as another program/command (a shell tool, a compiler, `git`, `ffmpeg`, whatever) - `subprocess` lets python start it and talk to it
* `subprocess.run([...], capture_output=True, text=True)` runs a command and gives us its output back as a nice string (`text=True`) instead of raw bytes
* **security note (same family of bug as SQL injection from lesson 13):** always pass the command as a LIST of separate arguments, never as one shell-interpreted string built with an f-string/`.format()`/`+` out of untrusted input - that pattern is called shell/command injection: if any piece of that string came from a user, they could sneak in things like `; rm -rf /` and suddenly run their OWN commands instead of just the value you intended. `shell=True` with concatenated strings is exactly the dangerous pattern to avoid

In [3]:
import subprocess  # standard library, lets python start and control other programs/commands
import sys

# example 1: ask the currently running python interpreter for its version
# note the LIST of arguments: [sys.executable, "--version"] - each piece is its own list item, nothing gets glued together as one string
result = subprocess.run([sys.executable, "--version"], capture_output=True, text=True)
print("returncode:", result.returncode)  # 0 almost always means "success" by unix convention, non-zero means something went wrong
print("stdout:", result.stdout.strip())  # capture_output=True catches what the command printed, text=True gives us a str instead of bytes

# example 2: a simple, safe command everyone has on a linux machine
result = subprocess.run(["echo", "hello from a subprocess"], capture_output=True, text=True)
print("stdout:", result.stdout.strip())
print("returncode:", result.returncode)

# DANGEROUS pattern - shown only as a comment, never actually run:
# untrusted_name = "Alex; rm -rf /"                                  # imagine this came from user input
# subprocess.run(f"echo hello {untrusted_name}", shell=True)          # shell=True + string concatenation --> command injection!
# the semicolon would end the "echo" command early and start a brand new, attacker-controlled command
# the list-based version above (subprocess.run(["echo", "hello", untrusted_name])) is immune to this - the whole value is always just ONE argument, never re-parsed as shell syntax

returncode: 0
stdout: Python 3.11.15
stdout: hello from a subprocess
returncode: 0


## file/folder automation - organizing files by extension
* one of the most common real-world automation requests: "clean up my messy Downloads folder" - sort files into subfolders by their extension automatically
* we use `pathlib` (recap from lesson 05): `.glob()` to find files, `.suffix` to read the extension, and `shutil.move()` to actually move them
* **important:** we only ever create/move files inside our own dedicated `automation_demo` folder here, so this lesson never touches anything real on disk - we clean the whole folder up again at the very end

In [4]:
from pathlib import Path
import shutil  # standard library, higher-level file operations than the raw os module (copying, moving, deleting whole trees)

# our dedicated demo folder - EVERYTHING file-related in this lesson happens inside here, nowhere else
demo_dir = Path("automation_demo")
demo_dir.mkdir(exist_ok=True)  # exist_ok=True means "dont crash if it already exists", handy if we re-run this cell

# lets pretend this is a messy "Downloads" folder with a handful of different file types mixed together
messy_files = {
    "report.txt": "just a text report\n",
    "notes.txt": "some quick notes\n",
    "data.csv": "name,score\nAlex,42\n",
    "backup.log": "2026-01-01 all good\n",
}
for filename, content in messy_files.items():
    (demo_dir / filename).write_text(content)  # pathlib convenience write, no need for a full "with open(...)" block for small text files

print("files before organizing:", sorted(p.name for p in demo_dir.glob("*")))

files before organizing: ['backup.log', 'data.csv', 'notes.txt', 'report.txt']


In [5]:
# now the actual "organize by extension" automation
# .glob("*") gives us every direct child (files AND folders) of demo_dir, we only want the files
for file_path in demo_dir.glob("*"):
    if not file_path.is_file():
        continue  # skip anything that is already a folder (like a subfolder we created in a previous run)

    extension = file_path.suffix.lstrip(".")  # ".txt" --> "txt", lstrip removes the leading dot so the folder name looks cleaner
    if not extension:
        continue  # a file with no extension at all - nothing sensible to sort it by, so we just leave it where it is

    target_folder = demo_dir / extension  # e.g. automation_demo/txt
    target_folder.mkdir(exist_ok=True)     # create the per-extension folder if it doesnt exist yet

    shutil.move(str(file_path), str(target_folder / file_path.name))  # actually move the file into its new home
    print(f"moved {file_path.name} --> {target_folder}/")

# lets look at the resulting folder structure
print("\nfolders after organizing:", sorted(p.name for p in demo_dir.glob("*") if p.is_dir()))
for subfolder in sorted(demo_dir.glob("*")):
    if subfolder.is_dir():
        print(subfolder, "contains:", [p.name for p in subfolder.glob("*")])

moved report.txt --> automation_demo/txt/
moved data.csv --> automation_demo/csv/
moved notes.txt --> automation_demo/txt/
moved backup.log --> automation_demo/log/

folders after organizing: ['csv', 'log', 'txt']
automation_demo/csv contains: ['data.csv']
automation_demo/log contains: ['backup.log']
automation_demo/txt contains: ['report.txt', 'notes.txt']


## batch renaming files
* another extremely common everyday request: "rename all these files following one consistent pattern" (photo imports, exported reports, whatever)
* here we add a numbered prefix to every file inside one of our per-extension folders using `.rename()`
* the same idea works for way more complex patterns (dates, sequence numbers, replacing spaces with underscores, ...) - the core loop-and-rename shape stays the same

In [6]:
txt_folder = demo_dir / "txt"  # our "report.txt" and "notes.txt" ended up here from the previous step

# sorted() first, so the numbering is deterministic/predictable instead of depending on random filesystem order
for index, file_path in enumerate(sorted(txt_folder.glob("*.txt")), start=1):
    new_name = f"{index:02d}_{file_path.name}"          # e.g. "01_notes.txt" - :02d pads the number with a leading zero
    new_path = file_path.with_name(new_name)             # .with_name() builds a new Path in the SAME folder, just a different filename
    file_path.rename(new_path)                           # actually perform the rename on disk
    print(f"renamed {file_path.name} --> {new_name}")

print("\ntxt folder now contains:", sorted(p.name for p in txt_folder.glob("*")))

renamed notes.txt --> 01_notes.txt
renamed report.txt --> 02_report.txt

txt folder now contains: ['01_notes.txt', '02_report.txt']


## web scraping basics
* "scraping" means downloading a webpage's HTML and then extracting the specific bits of data you actually care about out of it
* just like lesson 14, we do NOT scrape a real external website here - a real site's HTML can change at any time, and depending on it would make this lesson unreliable (and possibly break in offline/CI environments)
* instead we spin up our OWN tiny local HTML server (`http.server` + `threading`, exact same reliable pattern as lesson 14's sandbox API) serving a small hand-written page, and scrape THAT
* **in real usage** you would point `requests.get(...)` at a real url instead - but always check that site's `robots.txt` and terms of service first, scraping is NOT always allowed, some sites explicitly forbid it or have rate limits/legal restrictions
* for parsing the HTML we use the third-party `BeautifulSoup` library (`pip install beautifulsoup4`, already installed for this lesson) - it turns the raw HTML text into a searchable tree of tags

In [7]:
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer

# a small, hand-written HTML page with a few clearly identifiable elements we can practice extracting
DEMO_HTML_PAGE = """
<html>
<head><title>Demo Shop</title></head>
<body>
    <h1>Welcome to the Demo Shop</h1>
    <ul class="products">
        <li class="product">Apple</li>
        <li class="product">Banana</li>
        <li class="product">Cherry</li>
    </ul>
    <table id="prices">
        <tr><th>item</th><th>price</th></tr>
        <tr><td>Apple</td><td>1.20</td></tr>
        <tr><td>Banana</td><td>0.80</td></tr>
    </table>
    <a href="https://example.org/about">About us</a>
    <a href="https://example.org/contact">Contact</a>
</body>
</html>
"""


class DemoPageHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        # no matter which path is requested, we just always serve the same demo page - good enough for our scraping practice
        body_bytes = DEMO_HTML_PAGE.encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "text/html")
        self.send_header("Content-Length", str(len(body_bytes)))
        self.end_headers()
        self.wfile.write(body_bytes)

    def log_message(self, format, *args):
        pass  # silence the default per-request console logging, keeps our notebook output clean (same trick as lesson 14)


# port 0 = "OS please pick a free port for me", same reasoning as lesson 14 - avoids clashing with anything already running
scrape_server = HTTPServer(("127.0.0.1", 0), DemoPageHandler)
scrape_port = scrape_server.server_address[1]

# daemon=True background thread, so it never blocks our notebook/kernel from exiting
scrape_thread = threading.Thread(target=scrape_server.serve_forever, daemon=True)
scrape_thread.start()

demo_page_url = f"http://127.0.0.1:{scrape_port}/"
print("local html sandbox running on", demo_page_url)

local html sandbox running on http://127.0.0.1:33563/


In [8]:
import requests
from bs4 import BeautifulSoup  # third-party, pip install beautifulsoup4 - parses HTML into a searchable tree

# step 1: download the raw html, exactly like any other requests.get() call from lesson 14
response = requests.get(demo_page_url)
response.raise_for_status()  # same defensive habit as lesson 14 - blow up loudly if something went wrong instead of silently parsing garbage

# step 2: hand the raw HTML text to BeautifulSoup, "html.parser" is pythons builtin parser (no extra install needed for the parser itself)
soup = BeautifulSoup(response.text, "html.parser")

# extraction technique 1: .find() for a single element, then .get_text() to pull out just the text (no html tags)
page_title = soup.find("h1")
print("h1 text:", page_title.get_text())

# extraction technique 2: .find_all() for EVERY matching element - here every <li class="product">
product_items = soup.find_all("li", class_="product")
product_names = [item.get_text() for item in product_items]
print("all products:", product_names)

h1 text: Welcome to the Demo Shop
all products: ['Apple', 'Banana', 'Cherry']


In [9]:
# extraction technique 3: CSS-selector based lookup with .select() - very handy if you already know CSS selectors from web development
# "#prices tr" means "every <tr> inside the element with id='prices'"
price_rows = soup.select("#prices tr")
for row in price_rows:
    cells = row.find_all(["td", "th"])           # grab both header cells (th) and data cells (td)
    texts = [cell.get_text() for cell in cells]
    print("row:", texts)

# extraction technique 4: pulling an ATTRIBUTE (not text) out of a tag - here the href of every link
links = soup.find_all("a")
for link in links:
    href = link["href"]                # tags behave like a dict for their attributes
    link_text = link.get_text()
    print(f"link text={link_text!r} --> href={href}")

row: ['item', 'price']
row: ['Apple', '1.20']
row: ['Banana', '0.80']
link text='About us' --> href=https://example.org/about
link text='Contact' --> href=https://example.org/contact


## scheduling / running things periodically
* everything above ran once, right now - but a LOT of real automation needs to run again and again (every hour, every night, every monday morning, ...)
* actually sleeping for hours or setting up a real scheduler doesnt make sense inside a notebook lesson, so this part is mostly conceptual - just know these options exist:
* the third-party `schedule` package gives you a simple, very readable in-python api, e.g. `schedule.every(10).minutes.do(job)` or `schedule.every().day.at("09:00").do(job)` - you then run a small loop that calls `schedule.run_pending()` every second or so, and it fires your jobs at the right time. its nice for quick scripts, but the python process itself has to stay running the whole time
* the more typical REAL solution for anything that must survive reboots and run reliably completely unattended is an OS-level scheduler instead:
  * **cron** (linux/mac) - a system service that runs a command on a schedule you define with a cron expression (e.g. `0 9 * * *` = every day at 9:00), configured with the `crontab -e` command, keeps running even if you are not logged in
  * **Task Scheduler** (windows) - the windows equivalent, a GUI (and a `schtasks` command-line tool) to run a program/script on a time-based or event-based trigger, also survives reboots and logouts
* rule of thumb: `schedule` for a quick script you keep running yourself, cron/Task Scheduler for anything that genuinely needs to run unattended and reliably

## quick comparison to other languages
* python and bash/shell scripting have historically been THE two most common "glue code / automate my computer" choices - but python has increasingly taken over a lot of territory that used to be bash-only, mainly because bash gets unreadable fast beyond simple sequential commands, while python gives you real data structures (lists/dicts, not just text streams), proper error handling (try/except, lesson 06) and a giant standard library
* **PowerShell** (windows) plays a very similar "OS automation" role to bash, but with a more object-oriented, almost python-like feel in places - its commands (cmdlets) pass around real structured objects instead of just plain text between pipes, which bash always does
* **Node.js/JavaScript** is a common alternative specifically for web-scraping-heavy automation, because of excellent headless-browser tooling like Puppeteer/Playwright - worth knowing Playwright also has a first-class python api (`playwright-python`), so you dont necessarily have to leave python for that either
* **Go and Rust** are increasingly popular for building small, single-binary CLI tools - they compile down to ONE executable file you can hand to someone who doesnt even have python (or any runtime) installed at all, which is a real tradeoff a plain python script doesnt have (a python script always needs a python interpreter present on the machine that runs it)

## Exercises
1. build an `argparse` CLI that takes a `--number` argument (`type=int`) and prints whether it is even or odd (recap the `%` operator from lesson 03); call it with `parse_args()` on an explicit fake args list, like we did above
2. use `subprocess.run([...], capture_output=True, text=True)` to run a safe command of your choice (a list of arguments, no `shell=True`!) and print its captured stdout
3. inside your `automation_demo` folder, create 3 new files with different extensions, then write a small script that automatically sorts them into per-extension subfolders (same idea as the "organize by extension" section above)
4. scrape the local sandbox HTML page again and use `BeautifulSoup` to print all the text inside every `<li class="product">` element

In [10]:
# TODO: solve exercise 1 here (argparse --number, even/odd)


# TODO: solve exercise 2 here (subprocess, safe command, print stdout)


# TODO: solve exercise 3 here (3 new files in automation_demo, sort by extension)


# TODO: solve exercise 4 here (scrape product <li> text again)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [11]:
# sample solution 1 - argparse CLI, even/odd
even_odd_parser = argparse.ArgumentParser(description="tells you if a number is even or odd")
even_odd_parser.add_argument("--number", type=int, required=True, help="the number to check")

fake_args = even_odd_parser.parse_args(["--number", "17"])
if fake_args.number % 2 == 0:
    print(f"{fake_args.number} is even")
else:
    print(f"{fake_args.number} is odd")

# sample solution 2 - subprocess, safe command, print stdout
result = subprocess.run(["echo", "automation is fun"], capture_output=True, text=True)
print("captured stdout:", result.stdout.strip())

# sample solution 3 - create 3 files with different extensions, sort them into per-extension subfolders
exercise_files = {
    "summary.txt": "a quick summary\n",
    "numbers.csv": "a,b\n1,2\n",
    "run.log": "started ok\n",
}
for filename, content in exercise_files.items():
    (demo_dir / filename).write_text(content)

for file_path in demo_dir.glob("*"):
    if not file_path.is_file():
        continue
    extension = file_path.suffix.lstrip(".")
    if not extension:
        continue
    target_folder = demo_dir / extension
    target_folder.mkdir(exist_ok=True)
    shutil.move(str(file_path), str(target_folder / file_path.name))

print("folders now:", sorted(p.name for p in demo_dir.glob("*") if p.is_dir()))

# sample solution 4 - scrape the product <li> elements again
response = requests.get(demo_page_url)
soup = BeautifulSoup(response.text, "html.parser")
product_texts = [item.get_text() for item in soup.find_all("li", class_="product")]
print("scraped product names:", product_texts)

17 is odd


captured stdout: automation is fun
folders now: ['csv', 'log', 'txt']
scraped product names: ['Apple', 'Banana', 'Cherry']


## cleanup - never leave demo files/servers behind
* everything file-related in this lesson lived inside `automation_demo/` - time to remove that whole folder again
* and our local scraping sandbox server is still running in the background - time to shut it down properly, exactly like we did in lesson 14

In [12]:
import shutil

# remove the ENTIRE demo folder and everything inside it, including all the subfolders we created
shutil.rmtree(demo_dir, ignore_errors=True)
print("automation_demo still exists?", demo_dir.exists())  # should print False now

# shut the scraping sandbox server down properly
scrape_server.shutdown()             # tells serve_forever() to stop its loop
scrape_thread.join(timeout=2)        # wait for the background thread to actually finish
print("scraping sandbox server thread still alive?", scrape_thread.is_alive())  # should print False

automation_demo still exists? False


scraping sandbox server thread still alive? False


* congratulation you finished this lesson

## what we learned
* building a real command-line tool with `argparse`: required vs optional arguments, default values, `type=int` conversion, and the free auto-generated `--help` message
* running external commands safely with `subprocess.run([...], capture_output=True, text=True)`, and why a LIST of arguments (never `shell=True` + string concatenation) protects you from command injection
* automating file/folder cleanup with `pathlib`: finding files with `.glob()`, reading `.suffix`, and sorting them into subfolders with `shutil.move()`
* batch-renaming a bunch of files following one consistent pattern with `.rename()`/`.with_name()`
* web scraping basics: spinning up a reliable local `http.server` sandbox (same pattern as lesson 14), then extracting data with `BeautifulSoup` via `.find()`, `.find_all()`, `.select()`, `.get_text()` and attribute access like `link["href"]`
* why real scraping needs a check of `robots.txt`/terms of service first - its not always allowed
* scheduling options: the in-python `schedule` package for quick scripts, vs OS-level cron (linux/mac) and Task Scheduler (windows) for anything that must run reliably unattended
* always cleaning up everything a script creates - demo folders (`shutil.rmtree`) and background servers/threads (`.shutdown()` + `.join()`)